## 9. PD Modeling — Logistic Regression Scorecard

In [ ]:
from sklearn.model_selection import train_test_split

feature_cols = [
    'grade_woe', 'home_ownership_woe',
    'int_rate', 'fico_range_low', 'dti', 'mort_acc',
    'loan_amnt', 'annual_inc', 'revol_util', 'loan_to_income', 'has_pub_rec',
]
target_col = 'default'

model_df = df_cleaned[feature_cols + [target_col]].dropna()
print(f"Modeling dataset shape: {model_df.shape}")
print(f"Default rate: {model_df[target_col].mean():.4f}")

X = model_df[feature_cols]
y = model_df[target_col]

# 70 / 15 / 15 stratified split
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.30, random_state=42, stratify=y)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.50, random_state=42, stratify=y_temp)

print(f"\nTrain:      {X_train.shape[0]:>8,} rows | Default rate: {y_train.mean():.4f}")
print(f"Validation: {X_val.shape[0]:>8,} rows | Default rate: {y_val.mean():.4f}")
print(f"Test:       {X_test.shape[0]:>8,} rows | Default rate: {y_test.mean():.4f}")

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.calibration import CalibratedClassifierCV, calibration_curve
from sklearn.metrics import roc_auc_score, brier_score_loss

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled   = scaler.transform(X_val)
X_test_scaled  = scaler.transform(X_test)

lr_model = LogisticRegression(max_iter=1000, random_state=42, solver='lbfgs', class_weight='balanced')
lr_model.fit(X_train_scaled, y_train)

print("Coefficients after scaling:")
print(pd.Series(lr_model.coef_[0], index=feature_cols).sort_values(ascending=False))

raw_probs_val = lr_model.predict_proba(X_val_scaled)[:, 1]
print(f"\nRaw AUC on Validation: {roc_auc_score(y_val, raw_probs_val):.4f}")

# Platt scaling
calibrated_model = CalibratedClassifierCV(lr_model, method='sigmoid', cv='prefit')
calibrated_model.fit(X_val_scaled, y_val)

cal_probs_val  = calibrated_model.predict_proba(X_val_scaled)[:, 1]
cal_probs_test = calibrated_model.predict_proba(X_test_scaled)[:, 1]

print(f"AUC Validation: {roc_auc_score(y_val, cal_probs_val):.4f}")
print(f"AUC Test:       {roc_auc_score(y_test, cal_probs_test):.4f}")
print(f"Brier (raw):        {brier_score_loss(y_val, raw_probs_val):.4f}")
print(f"Brier (calibrated): {brier_score_loss(y_val, cal_probs_val):.4f}")

# Calibration plot
fig, ax = plt.subplots(figsize=(8, 6))
ax.plot([0,1], [0,1], 'k--', label='Perfect')
fp_raw, mp_raw = calibration_curve(y_val, raw_probs_val, n_bins=10)
ax.plot(mp_raw, fp_raw, 'b-o', label='Raw LR')
fp_cal, mp_cal = calibration_curve(y_val, cal_probs_val, n_bins=10)
ax.plot(mp_cal, fp_cal, 'r-o', label='Platt Scaled')
ax.set_title('Calibration Plot — Raw vs Platt Scaled')
ax.legend()
plt.tight_layout()
plt.show()

**Result:** Logistic regression scorecard AUC = 0.70. Platt scaling improved Brier score from 0.22 to 0.15.

In [ ]:
# Industry standard score scaling: BASE_SCORE=600, PDO=20
BASE_SCORE = 600
PDO = 20
BASE_ODDS = 1/0.213  # 1 / average_default_rate
FACTOR = PDO / np.log(2)
OFFSET = BASE_SCORE - (FACTOR * np.log(BASE_ODDS))

def prob_to_score(prob, factor=FACTOR, offset=OFFSET):
    prob = np.clip(prob, 1e-6, 1 - 1e-6)
    return round(offset + factor * np.log((1 - prob) / prob))

test_scores = np.array([prob_to_score(p) for p in calibrated_model.predict_proba(X_test_scaled)[:, 1]])

print(f"Min: {test_scores.min()}  Max: {test_scores.max()}  Mean: {test_scores.mean():.0f}")

fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(test_scores[y_test == 0], bins=50, alpha=0.5, color='blue', label='Paid (Good)', density=True)
ax.hist(test_scores[y_test == 1], bins=50, alpha=0.5, color='red',  label='Default (Bad)', density=True)
ax.set_title('Credit Score Distribution — Default vs Paid')
ax.set_xlabel('Credit Score')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
from sklearn.metrics import roc_curve, classification_report, confusion_matrix, ConfusionMatrixDisplay

auc_test  = roc_auc_score(y_test, cal_probs_test)
fpr, tpr, _ = roc_curve(y_test, cal_probs_test)
ks_stat   = max(tpr - fpr)
gini_test = 2 * auc_test - 1

print(f"AUC: {auc_test:.4f}  |  Gini: {gini_test:.4f}  |  KS: {ks_stat:.4f}")

fig, ax = plt.subplots(figsize=(8, 6))
ax.plot(fpr, tpr, label=f'LR Scorecard (AUC={auc_test:.4f})')
ax.plot([0,1],[0,1],'k--')
ax.set_title('ROC Curve — Logistic Regression Scorecard')
ax.legend()
plt.tight_layout()
plt.show()

y_pred = (cal_probs_test >= 0.5).astype(int)
ConfusionMatrixDisplay(confusion_matrix(y_test, y_pred), display_labels=['Paid','Default']).plot(cmap='Blues')
plt.title('Confusion Matrix — Logistic Scorecard')
plt.show()
print(classification_report(y_test, y_pred, target_names=['Paid','Default']))